# Team games — BasketballStatsVlaanderen

Scrapes the game schedule/results table for a team page on `app.basketballstatsvlaanderen.be` into a pandas DataFrame.

The site is a SvelteKit app; the games table (`Tabulator` grid) is populated client-side and paginated, and the full list only renders once the **"Alle wedstrijden tonen"** (show all games) checkbox is checked — so this uses Playwright (headless Chromium) rather than a plain HTTP request.

Note: `IsStatsPublic` / `IsExtraStatsPublic` are `false` for this team, so only the schedule (date/time/opponent) is exposed — no box-score stats are available regardless of scraping method.

In [1]:
import re
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from playwright.sync_api import sync_playwright

In [2]:
TEAM_URL = "https://app.basketballstatsvlaanderen.be/clubs/BVBL1037/BVBL1037HSE%20%202?season=2627"
HEADLESS = True

In [3]:
def scrape_team_games(url: str, headless: bool = True) -> tuple[str, pd.DataFrame]:
    """Return (team_name, games_df) for a team page URL, across all pages of the games table.

    Runs its own asyncio event loop internally (via Playwright's sync API), so this must be
    called from a worker thread, not Jupyter's main thread -- see the call site below.
    """
    import asyncio

    # ipykernel sets the process-wide policy to WindowsSelectorEventLoopPolicy (needed for zmq),
    # but Playwright's driver subprocess requires WindowsProactorEventLoopPolicy. Swap it just
    # for the duration of this call, in this worker thread, then restore it.
    original_policy = asyncio.get_event_loop_policy()
    if hasattr(asyncio, "WindowsProactorEventLoopPolicy"):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    try:
        rows_out = []
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=headless)
            page = browser.new_page()
            page.goto(url, wait_until="networkidle", timeout=30000)
            page.wait_for_timeout(1000)

            team_name = page.locator("h1.page-title").inner_text().strip()
            team_name = re.sub(r"\s*\n.*", "", team_name).strip()

            # Full season list (incl. past games) is hidden behind this checkbox by default
            show_all = page.locator("#showAllGames")
            if show_all.count():
                show_all.check()
                page.wait_for_timeout(1500)

            while True:
                for row in page.locator(".tabulator-row").all():
                    cell = lambda field: row.locator(f'[tabulator-field="{field}"]').inner_text().strip()
                    href = row.locator('[tabulator-field="Date"] a').get_attribute("href") or ""
                    guid = href.split("/games/")[-1] if "/games/" in href else None
                    rows_out.append({
                        "Guid": guid,
                        "Date": cell("Date"),
                        "Time": cell("Time"),
                        "HomeTeam": cell("HomeTeam"),
                        "AwayTeam": cell("AwayTeam"),
                        "Result": cell("Result"),
                    })

                next_btn = page.locator('.tabulator-page[data-page="next"]')
                if next_btn.count() == 0 or next_btn.get_attribute("disabled") is not None:
                    break
                next_btn.click()
                page.wait_for_timeout(800)

            browser.close()
    finally:
        asyncio.set_event_loop_policy(original_policy)

    df = pd.DataFrame(rows_out, columns=["Guid", "Date", "Time", "HomeTeam", "AwayTeam", "Result"])
    if not df.empty:
        df["Opponent"] = df.apply(lambda r: r["AwayTeam"] if r["HomeTeam"] == team_name else r["HomeTeam"], axis=1)
        df["IsHome"] = df["HomeTeam"] == team_name
    return team_name, df

In [4]:
# Jupyter's kernel already runs an asyncio event loop, which conflicts with
# Playwright's sync API if called directly -> run it in a plain worker thread instead.
with ThreadPoolExecutor(max_workers=1) as executor:
    team_name, games_df = executor.submit(scrape_team_games, TEAM_URL, HEADLESS).result()

print(f"Team: {team_name}")
print(f"Games found: {len(games_df)}")
games_df

Team: BBC Haantjes Certifisc Oudenaarde HSE B
Games found: 30


,Guid,Date,Time,HomeTeam,AwayTeam,Result,Opponent,IsHome
0,BVBL26271124OR00120708,2026-08-16,14:00,Mibac Middelkerke HSE B,BBC Haantjes Certifisc Oudenaarde HSE B,,Mibac Middelkerke HSE B,False
1,BVBL26271037OR00040511,2026-08-26,20:30,BBC Haantjes Certifisc Oudenaarde HSE B,Basket Desselgem HSE B,,Basket Desselgem HSE B,True
2,BVBL26271037OR00040505,2026-08-29,18:15,BBC Haantjes Certifisc Oudenaarde HSE B,Blue Rocks Ronse-Kluisbergen HSE B,,Blue Rocks Ronse-Kluisbergen HSE B,True
3,BVBL26279130BOVHSEVR04,2026-09-05,20:45,Erembodegem Silverbacks HSE A,BBC Haantjes Certifisc Oudenaarde HSE B,,Erembodegem Silverbacks HSE A,False
4,BVBL26279130OVHSE31ABD,2026-09-12,20:15,BBC Haantjes Certifisc Oudenaarde HSE B,BBC Lokeren HSE A,,BBC Lokeren HSE A,True
5,BVBL26279130OVHSE31AEB,2026-09-19,17:00,Gent-Oost Eagles HSE D,BBC Haantjes Certifisc Oudenaarde HSE B,,Gent-Oost Eagles HSE D,False
6,BVBL26279130OVHSE31ABG,2026-09-26,20:15,BBC Haantjes Certifisc Oudenaarde HSE B,Erembodegem Silverbacks HSE A,,Erembodegem Silverbacks HSE A,True
7,BVBL26279130OVHSE31AIB,2026-10-03,21:00,KBBC Eksaarde HSE A,BBC Haantjes Certifisc Oudenaarde HSE B,,KBBC Eksaarde HSE A,False
8,BVBL26279130OVHSE31ABK,2026-10-10,20:15,BBC Haantjes Certifisc Oudenaarde HSE B,KBBC Sparta Laarne HSE C,,KBBC Sparta Laarne HSE C,True
9,BVBL26279130OVHSE31AAB,2026-10-17,18:30,BBC Helios SanoRice Zottegem HSE B,BBC Haantjes Certifisc Oudenaarde HSE B,,BBC Helios SanoRice Zottegem HSE B,False


**Note:** `season=2526` (2025-2026) and `season=2425` (2024-2025) currently return **0 games** for this team — the site itself has no games recorded for those seasons (confirmed directly in the rendered page, not a scraping issue). `season=2627` (2026-2027, the current season) has the full 29-game schedule, which is what `TEAM_URL` above now points to. Change the `season` value or the club/team GUIDs in `TEAM_URL` to pull a different season or team.

In [5]:
# Optional: save to CSV alongside this notebook
# games_df.to_csv("team_games.csv", index=False)